# 🎙️ Run Haru-4B on Google Colab / Kaggle (100% Free GPU)

This notebook lets you run **[higurashi-ai/Haru-4B](https://huggingface.co/higurashi-ai/Haru-4B)** (the continuous full-duplex spoken dialogue foundation model) and its **[GGUF Quantized Release](https://huggingface.co/Jayanth-MKV/Haru-4B-GGUF)** on **Google Colab** (Free T4 GPU) or **Kaggle** (Free T4/P100 GPU).

### ⚡ Hardware Requirements:
- **Google Colab:** Go to `Runtime` -> `Change runtime type` -> Select **T4 GPU** (15 GB VRAM).
- **Kaggle:** Go to Settings (right panel) -> `Accelerator` -> Select **GPU T4 x2** or **GPU P100**.

In [ ]:
# Step 1: Verify Free Cloud GPU (Must show Tesla T4 or P100 with >= 15GB VRAM)
!nvidia-smi

In [ ]:
# Step 2: Clone Haru Repository & Install Dependencies
!git clone https://github.com/higurashi-ai/Haru.git
%cd Haru
!pip install -q -e .
!pip install -q soundfile torchaudio transformers accelerate huggingface_hub

In [ ]:
# Step 3: Install cloudflared to expose the Web Audio Portal
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start cloudflared background tunnel on port 8998
import subprocess, time, re
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8998"], 
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
start_time = time.time()
while time.time() - start_time < 15:
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("=" * 60)
if public_url:
    print(f"🎉 PUBLIC WEB PORTAL URL: {public_url}")
else:
    print("Tunnel started. Check output or use Colab local server.")
print("=" * 60)

In [ ]:
# Step 4: Launch Haru Full-Duplex Server
# Note: --warmup-frames 0 bypasses slow startup computation
!python -m haru.server --host 0.0.0.0 --port 8998 --warmup-frames 0

---
## 🧪 Option B: Run Quantized GGUF Backbone via llama.cpp

You can also run our public quantized release [Jayanth-MKV/Haru-4B-GGUF](https://huggingface.co/Jayanth-MKV/Haru-4B-GGUF) using GPU acceleration (`-ngl 99` offloads all 32 layers to GPU VRAM for >60 tokens/sec):

In [ ]:
# 1. Download precompiled llama.cpp binary with CUDA support
!git clone https://github.com/ggml-org/llama.cpp.git
%cd llama.cpp
!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -j 4

# 2. Download our Quantized Haru-4B GGUF Model (2.6 GB)
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(repo_id="Jayanth-MKV/Haru-4B-GGUF", filename="qwen-haru-q4_k_m.gguf")
print("Downloaded GGUF to:", model_path)

# 3. Run high-speed inference with 100% GPU offload
!./build/bin/llama-cli -m "{model_path}" -p "Hello! How are you doing today?" -n 128 -ngl 99